### Comparing Aggregate Models for Classification

This approach used the VotingClassifier wisdom of the crowds with "soft" voting to choose the classifier from Logistic Regression, 
Random Forest, SVM (RBF), Gradient Boosting  K-Nearest Neighbors, LDA, and Naive Bayes. The parameters chosen were 'Age', 'MMSE', 'CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)', 'CSF Phosphorylated tau (pg/mL)', 'Sex', and 'APOE4'.  Sorting by procision, then roc_auc, RandomForestClassifier had the following scores:

```
Pipeline([('clf', RandomForestClassifier(n_estimators=300, random_state=42)),
])
```
|Accuracy|Precision|Recall|F2|AUC|
|--------|---------|------|--|-------|
|0.777778|0.875000|0.7|0.729167|0.73750|


Hyper-parameter tuning usign GridSearhCV produced the parameter 'clf__class_weight': None, 'clf__max_depth': 3, 'clf__min_samples_leaf': 1

```
Pipeline([('clf', RandomForestClassifier(n_estimators=300, 
                                   random_state=42,
                                   class_weight=None,
                                   max_depth=3,
                                   min_samples_leaf=1,
        )),
])
```
|Accuracy|Precision|Recall|F2|AUC|false_negatives| false negative rate |
|--------|---------|------|--|-------|----|----|
|0.777778|0.875000|0.7|0.729167|0.73750|3| 0.3|




accuracy': 0.7222222222222222, 
'precision': 0.7777777777777778, 
'auc': 0.8875, 
'false_negatives': 3, 
'false_negative_rate': np.float64(0.3)

In [14]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
import seaborn as sns
sns.set_palette("pastel")
import plotly.express as px
import matplotlib.pyplot as plt
import math
import numpy as np

from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline

pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
%matplotlib inline

In [15]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, make_scorer
)
from sklearn.inspection import permutation_importance

In [16]:
# 1. Load data, keep only patients with a known outcome (MCI patients)
df = pd.read_csv("data/plasma_lipidomics.csv")
mci = df[df["Progression to Alzheimer's Disease"].notna()].copy()

# 2. Fill missing numeric values with the column median
numeric_cols = ['Age', 'MMSE', 'CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)',
                 'CSF Phosphorylated tau (pg/mL)']
for col in numeric_cols:
    median_val = mci[col].median()
    mci[col] = mci[col].fillna(median_val)

# 3. Fill missing categorical values with the most common value
mode_val = mci['APOE4'].mode()[0]
mci['APOE4'] = mci['APOE4'].fillna(mode_val)

# 4. Convert categorical text columns to numeric (0/1)
mci['Sex'] = (mci['Sex'] == 'Male').astype(int)          # Male=1, Female=0
mci['APOE4'] = (mci['APOE4'] == 'Yes').astype(int)        # carries APOE4 allele=1, no=0
mci['Target'] = (mci["Progression to Alzheimer's Disease"] == 'Yes').astype(int)
#df = mci.copy()
#df = df.rename(columns={'CSF Amyloid (pg/mL)': 'amyloid', 'CSF Phosphorylated tau (pg/mL)': 'tau', "Progression to Alzheimer's Disease" : 'progression'})

In [17]:
# Display the first few rows to understand the data
mci.head()

,Sample,Diagnostic,Sex,Age,MMSE,CSF Amyloid (pg/mL),CSF Total tau (pg/mL),CSF Phosphorylated tau (pg/mL),APOE4,Progression to Alzheimer's Disease,Progression time (months),Target
64,65,Mild Cognitive Impairment,1,69,23,595.0,465.0,75.0,0,No,NaN,0
104,105,Mild Cognitive Impairment,1,70,27,1845.0,353.0,92.4,0,No,NaN,0
105,106,Mild Cognitive Impairment,0,73,29,928.0,531.0,176.0,0,No,NaN,0
106,107,Mild Cognitive Impairment,0,68,23,619.0,477.0,142.0,1,Yes,14.0,1
107,108,Mild Cognitive Impairment,1,78,27,784.0,1231.0,394.0,0,No,NaN,0


## Setting up the model parameters

In [18]:
# Build the modeling frame: predict Target (Progression to Alzheimer's Disease: 1 = Yes, 0 = No)
# from CSF biomarkers, MMSE cognitive score, and demographics
model_df = mci[['Age', 'MMSE', 'CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)', 'CSF Phosphorylated tau (pg/mL)',  'Sex', 'APOE4', 'Target']].copy()
model_df = model_df.rename(columns={
    'Age': 'age',
    'MMSE': 'mmse',
    'CSF Amyloid (pg/mL)': 'amyloid',
    'CSF Phosphorylated tau (pg/mL)': 'tau',
    'CSF Total tau (pg/mL)': 'total_tau',
    'Sex': 'sex',
    'APOE4': 'apoe4'
})

# already cleaned and imputed above; no missing values remain
model_df.columns

Index(['age', 'mmse', 'amyloid', 'total_tau', 'tau', 'sex', 'apoe4', 'Target'], dtype='str')

## Separate for Training

In [19]:
# Separate features and target variable
X = model_df.drop(columns=['Target'])
y = model_df['Target']

# sex and apoe4 are already 0/1 encoded, no dummy variables needed

# Split the data into training and testing sets (stratify to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## Models

In [20]:
# Define individual classifiers
models = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
    'Random Forest':        RandomForestClassifier(n_estimators=300, random_state=42),
    'SVM (RBF)':            Pipeline([('sc', StandardScaler()), ('clf', SVC(probability=True, random_state=42))]),
    'Gradient Boosting':    GradientBoostingClassifier(random_state=42),
    'K-Nearest Neighbors':  Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier())]),
    'LDA':                  Pipeline([('sc', StandardScaler()), ('clf', LinearDiscriminantAnalysis())]),
    'Naive Bayes':          Pipeline([('sc', StandardScaler()), ('clf', GaussianNB())]),
}

# Define the Voting Classifier (soft voting: every base model supports predict_proba)
voting_clf = VotingClassifier(estimators=list(models.items()), voting='soft')

# Function to evaluate models. We're concerned about false negatives (missed progressors),
# so we track recall and the F2 score (weights recall twice as heavily as precision)
# alongside accuracy, precision, F1, and ROC-AUC rather than reporting accuracy alone.
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'f2': fbeta_score(y_test, y_pred, beta=2, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'false_negatives': int(fn),
        'false_negative_rate': fnr,
    }

## GridSearchCV

## Coefficients

In [24]:
# Feature importance from Random Forest
rf_model = models['Random Forest']
rf_model.fit(X_train, y_train)
importances_rf = rf_model.feature_importances_
feature_importance_rf_df = pd.DataFrame({'feature': X_train.columns, 'importance': importances_rf})
feature_importance_rf_df = feature_importance_rf_df.sort_values(by='importance', ascending=False)
print("\nFeature Importance from Random Forest:")
print(feature_importance_rf_df)



Feature Importance from Random Forest:
     feature  importance
2    amyloid    0.242883
3  total_tau    0.205625
4        tau    0.175676
6      apoe4    0.128025
0        age    0.109455
1       mmse    0.095429
5        sex    0.042907


In [35]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import make_scorer, fbeta_score, precision_score, accuracy_score, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

best_features = ['amyloid', 'total_tau']
X_best = X[best_features]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline([
    ('clf', RandomForestClassifier(n_estimators=300, random_state=42)),
])

# RandomForest hyperparameters — modest grid given n=89
param_grid = {
    'clf__max_depth': [3, 5, None],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__class_weight': [None, 'balanced'],
}

f2_scorer = make_scorer(fbeta_score, beta=2, zero_division=0)
scoring_options = {
    'recall': 'recall',
    'f2': f2_scorer,
}

def run_grid_search(scoring_name, scoring):
    grid = GridSearchCV(pipe, param_grid, scoring=scoring, cv=cv, n_jobs=1)
    grid.fit(X_best, y)
    best_model = grid.best_estimator_

    y_pred = cross_val_predict(best_model, X_best, y, cv=cv, method='predict')
    y_proba = cross_val_predict(best_model, X_best, y, cv=cv, method='predict_proba')[:, 1]

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

    return {
        'scoring': scoring_name,
        'best_params': grid.best_params_,
        'accuracy': accuracy_score(y, y_pred),       
        'precision': precision_score(y, y_pred),
        'auc': roc_auc_score(y, y_proba),
        'false_negatives': int(fn),
        'false_negative_rate': fnr,
    }

results = [run_grid_search(name, scoring) for name, scoring in scoring_options.items()]
comparison_df = pd.DataFrame(results)
print(comparison_df)

  scoring  \
0  recall   
1      f2   

                                                                    best_params  \
0  {'clf__class_weight': None, 'clf__max_depth': 3, 'clf__min_samples_leaf': 1}   
1  {'clf__class_weight': None, 'clf__max_depth': 3, 'clf__min_samples_leaf': 1}   

   accuracy  precision       auc  false_negatives  false_negative_rate  
0  0.775281   0.813953  0.800152               12             0.255319  
1  0.775281   0.813953  0.800152               12             0.255319  


In [28]:
comparison_df['best_params']

0    {'clf__class_weight': None, 'clf__max_depth': 3, 'clf__min_samples_leaf': 1}
1    {'clf__class_weight': None, 'clf__max_depth': 3, 'clf__min_samples_leaf': 1}
Name: best_params, dtype: object

In [36]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, roc_auc_score
)
import pandas as pd

best_features = ['amyloid', 'total_tau']
X_best = X[best_features]

# ---- Stratified train/test split (80/20), preserves class balance ----
X_train, X_test, y_train, y_test = train_test_split(
    X_best, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train size: {len(X_train)}  ({y_train.sum()} Yes / {len(y_train) - y_train.sum()} No)")
print(f"Test size:  {len(X_test)}  ({y_test.sum()} Yes / {len(y_test) - y_test.sum()} No)")

# ---- Build and fit the final model ----

final_model = Pipeline([
    ('clf', RandomForestClassifier(n_estimators=300, 
                                   random_state=42,
                                   class_weight=None,
                                   max_depth=3,
                                   min_samples_leaf=1,
                                  )),
])

final_model.fit(X_train, y_train)

# ---- Evaluate on the held-out test set ----
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print(pd.DataFrame(cm, index=['Actual: No', 'Actual: Yes'], columns=['Pred: No', 'Pred: Yes']))
print(classification_report(y_test, y_pred, target_names=['No progression', 'Progressed'], digits=3))

tn, fp, fn, tp = cm.ravel()
fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

test_result = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'auc': roc_auc_score(y_test, y_proba),
    'false_negatives': int(fn),
    'false_negative_rate': fnr,
}
print(test_result)

Train size: 71  (37 Yes / 34 No)
Test size:  18  (10 Yes / 8 No)
             Pred: No  Pred: Yes
Actual: No          6          2
Actual: Yes         3          7
                precision    recall  f1-score   support

No progression      0.667     0.750     0.706         8
    Progressed      0.778     0.700     0.737        10

      accuracy                          0.722        18
     macro avg      0.722     0.725     0.721        18
  weighted avg      0.728     0.722     0.723        18

{'accuracy': 0.7222222222222222, 'precision': 0.7777777777777778, 'auc': 0.775, 'false_negatives': 3, 'false_negative_rate': np.float64(0.3)}


```
'accuracy': 0.7222222222222222, 
'precision': 0.7777777777777778, 
'auc': 0.8875, 
'false_negatives': 3, 
'false_negative_rate': np.float64(0.3)
```

In [38]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import make_scorer, fbeta_score, precision_score, accuracy_score, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

best_features = ['amyloid', 'total_tau']
X_best = X[best_features]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

final_model = Pipeline([
    ('clf', RandomForestClassifier(n_estimators=300, 
                                   random_state=42,
                                   class_weight=None,
                                   max_depth=3,
                                   min_samples_leaf=1,
                                  )),
])

# RandomForest hyperparameters — modest grid given n=89
param_grid = {
    'clf__max_depth': [3, 5, None],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__class_weight': [None, 'balanced'],
}

f2_scorer = make_scorer(fbeta_score, beta=2, zero_division=0)
scoring_options = {
    'recall': 'recall',
    'f2': f2_scorer,
}

def run_grid_search(scoring_name, scoring):
    grid = GridSearchCV(pipe, param_grid, scoring=scoring, cv=cv, n_jobs=1)
    grid.fit(X_best, y)
    best_model = grid.best_estimator_

    y_pred = cross_val_predict(best_model, X_best, y, cv=cv, method='predict')
    y_proba = cross_val_predict(best_model, X_best, y, cv=cv, method='predict_proba')[:, 1]

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

    return {
        'scoring': scoring_name,
        'best_params': grid.best_params_,
        'accuracy': accuracy_score(y, y_pred),       
        'precision': precision_score(y, y_pred),
        'auc': roc_auc_score(y, y_proba),
        'false_negatives': int(fn),
        'false_negative_rate': fnr,
    }

results = [run_grid_search(name, scoring) for name, scoring in scoring_options.items()]
comparison_df = pd.DataFrame(results)
print(comparison_df)

  scoring  \
0  recall   
1      f2   

                                                                    best_params  \
0  {'clf__class_weight': None, 'clf__max_depth': 3, 'clf__min_samples_leaf': 1}   
1  {'clf__class_weight': None, 'clf__max_depth': 3, 'clf__min_samples_leaf': 1}   

   accuracy  precision       auc  false_negatives  false_negative_rate  
0  0.775281   0.813953  0.800152               12             0.255319  
1  0.775281   0.813953  0.800152               12             0.255319  
